# Análisis del Enunciado: Fraud Prevention Fintech – MercadoLibre Data Scientist Challenge

En el siguiente notebook, se realiza un análisis previo al enunciado especificado `./enunciado/Enunciado [Español].pdf`, se explica a continuación, el enfoque metodológico y técnico aplicado a la solución esperada, además, de referencias bibliográficas que sustentan el análisis realizado

## Contexto del Problema

MercadoLibre procesa miles de transacciones por segundo. El fraude en plataformas de pago representa pérdidas económicas a gran escala. El desafío propone construir un sistema de ML para **predecir transacciones fraudulentas**, con restricciones económicas específicas:

| Concepto | Valor |
|---|---|
| Margen de ganancia por transacción legítima aprobada | **25%** del monto |
| Pérdida por transacción fraudulenta aprobada | **100%** del monto |
| Variable objetivo | `fraude` (binaria: 0 = legítima, 1 = fraude) |

***

## Modelo que Maximiza la Ganancia de la Empresa

> *Entrene un algoritmo con el fin de predecir el fraude sabiendo que por cada transacción el porcentaje de ganancia es de un 25%, y por cada fraude aprobado se pierde el 100% del dinero de la transacción. Realizar un análisis y determinar un modelo que permita maximizar la ganancia de la empresa.*

**Según lo anterior podemos suponer lo siguiente:**

- Tener en cuenta que no se refiere a un problema de **clasificación estándar**, sino de **optimización de ganancia bajo una función de costo**, según lo enunciado en el punto anterior, esta es la clave para realizar un modelo de ML futuro que satisfaga la necesidad de optimizar la ganancia por cada transacción aprobada

### Matriz de confusión y costos por predicción

Como cualqueir problema de clasificación debemos definir la matriz de confusión asociada, teniendo en cuenta lo que se busca optimizar

| | **Predicción: Fraude (Bloquear)** | **Predicción: Legítima (Aprobar)** |
|---|---|---|
| **Real: Fraude** | **TP** – Fraude bloqueado | **FN** – Fraude aprobado |
| **Real: Legítima** | **FP** – Legítima bloqueada | **TN** – Legítima aprobada |

- **TP**: Se evita el 100% de pérdida. **Costo = 0**
- **FP**: Costo de oportunidad = **–25% del monto**
- **FN**: Pérdida = **–100% del monto**
- **TN**: Ganancia = **+25% del monto**

**Nota**
- Un **Falso positivo**, es aquella transacción que el modelo predijo como fraude pero que no lo era, esto hace que se pierda la ganancia del 25% por cada transacción aprobada, y en caso de un **Falso Negativo**, se pierde la detección del fraude, es decir, hay 100% de costo de perdida, estos `FP` y `FN`, son los únicos que modifican el estado de ganancia de la empresa


### Cost-Sensitive Learning

Según el enunciado principal, se puede afirmar que un acierto o un error en la aprobación o rechazo de una transacción no cuestan lo mismo, para este tipo de problemas dónde no solo se debe de resolver un problema de clasifdicación, es necesario aplicar técnicas adiconales como el `Cost-Sensitive Learning`, este enfoque de ML, reconoce que no todos los errores cuestan lo mismo, lo cual permite entrenar un modelo optimizando una ganancia según el peso de un acierto o no, en lugar del enfoque tradicional que trata cada acierto y error por igual. 

En los siguientes escenarios se puede observar al detalle porque una transacción puede tener una implicación en la ganancia de manera diferente

- Escenario 1 Transacción legítima (TN): el cliente paga J (valor del pago) La empresa se queda con el 25% de comisión/margen, $$\text{Beneficio} = +0.25 \times J$$
- Escenario 2 Aprobar fraude (FN): La transacción es fraudulenta se debe devolver el 100% del monto $$\text{Beneficio} = -1.00 \times J$$
- Escenario 3 Rechazar un fraude (TP): se captura el fraude y por ende la empresa aunque no gana detecta o protege al usuario de un fraude, no hay ganancia ni pérdida operativa $$\text{Beneficio} = 0$$
- Escenario 4: Rechazar transaccion legítima (FP): No se pierde dinero, pero tampoco se gana la comisión, acá aunque no hay ganancia neta, se puede considerar que hay un posible costo de oportunidad de $$\text{Oportunidad} = -0.25 \times J$$

**Nota**:
- En cuanto a un `FP` y un `FN`, la pérdida de ganancia no es igual, es decir, al tener un `FN` se pierde el 100%, pero con un `FP` solo el 25% de ganancia que hubieramos obtenido si la trasnacción se hubiese aprobado, en este caso, un `FN` cuesta 4 veces más que un `FP`, es decir, tenemos una relación de pérdida **4:1** respecto a un `FN` y un `FP`

Basados en lo mencionado en **The Foundations of Cost-Sensitive Learning", Charles Elkan (2001)** se menciona cómo tomar decisiones óptimas cuando cada tipo de error tiene un costo distinto, usando una matriz de costos, además define cuándo una matriz de costos es "razonable" (coherente económicamente), como en este caso, dónde al inico del probelma e define el peso del fraude y no fraude a nivel de beneficio

### Función de ganancia

#### Paso 1 — Identificar el impacto financiero de cada resultado

Con base a lo anterior, y la matriz de costos se identifica cada acierto o error:

- **TP**: Se evita el 100% de pérdida. **Costo = 0**
- **FP**: Costo de oportunidad = **–25% del monto**
- **FN**: Pérdida = **–100% del monto**
- **TN**: Ganancia = **+25% del monto**

#### Paso 2 — Función de Ganancia general

Según el impacto de cada transacción, se peude definir la siguiente ecuación que representa la ganancia:

$$J_{\text{general}} = (+0.25) \cdot TN \;+\; (-1.00) \cdot FN \;+\; (-0.25) \cdot FP \;+\; (0) \cdot TP$$

$$\boxed{J_{\text{general}} = 0.25 \cdot TN - 1.00 \cdot FN - 0.25 \cdot FP}$$

> **$J_{\text{general}}$** es la ganancia financiera real y total que produce el modelo para la empresa. Depende de los cuatro cuadrantes de la matriz de confusión.

#### Paso 3 — Identificar qué parte es controlable por el modelo

El modelo no puede cambiar cuántas transacciones fraudulentas o legítimas existen en el mundo. Lo que sí controla es **cómo clasifica cada una**. Por eso separamos los términos fijos de los variables.

El total de transacciones legítimas en el dataset es una cantidad fija:

$$N_{\text{legítimas}} = TN + FP = \text{constante del dataset}$$

> **$N_{\text{legítimas}}$** es el conteo total de transacciones legítimas presentes en el conjunto de datos

De esta identidad se obtiene que:

$$TN = N_{\text{legítimas}} - FP$$

#### Paso 4 — Sustituir TN y factorizar la constante

Reemplazando $TN$ en la ecuación de $J_{\text{general}}$:

$$J_{\text{general}} = 0.25 \cdot (N_{\text{legítimas}} - FP) - 1.00 \cdot FN - 0.25 \cdot FP$$

Distribuyendo el $0.25$:

$$J_{\text{general}} = 0.25 \cdot N_{\text{legítimas}} - 0.25 \cdot FP - 1.00 \cdot FN - 0.25 \cdot FP$$

Agrupando los términos con FP:

$$J_{\text{general}} = \underbrace{0.25 \cdot N_{\text{legítimas}}}_{C} - 1.00 \cdot FN - 0.25 \cdot FP$$

> **$C = 0.25 \times N_{\text{legítimas}}$** es una constante fija: es la ganancia máxima teórica que la empresa obtendría si aprobara **todas** las transacciones legítimas y ninguna fuera fraude. Es independiente del modelo porque $N_{\text{legítimas}}$ no cambia.  

#### Paso 5 — Definir la función de optimización J

Como $C$ es constante, maximizar $J_{\text{general}}$ es idéntico a maximizar solo la parte variable:

$$J_{\text{general}} = C + J_{\text{optima}} \quad \Longrightarrow \quad \max J_{\text{general}} \equiv \max J$$

donde:

$$\boxed{J_{\text{optima}} = -1.00 \cdot FN - 0.25 \cdot FP}$$

> Esta es la función de optimización del modelo. Mide únicamente el **daño financiero causado por los errores del modelo**, ignorando la constante $C$ que el modelo no puede controlar. Su valor es siempre $\leq 0$: cuanto más cercano a cero, mejor el modelo. Si el modelo fuera perfecto ($FN = 0$, $FP = 0$), entonces $J = 0$.

### 1.4 Umbral de Decisión Óptimo Teórico

Tambien puede definirse un threshold óptimo de decisión, en **The Foundations of Cost-Sensitive Learning", Charles Elkan (2001)** se establece que, dado el ratio de costos, el threshold óptimo de clasificación es:

$$p^* = \frac{Cost(FP)}{Cost(FP) + Cost(FN)} = \frac{0.25}{0.25 + 1.00} = \frac{0.25}{1.25} = \mathbf{0.20}$$

> **Interpretación:** Si el modelo estima que una transacción tiene probabilidad de fraude **≥ 20%**, debe bloquearse. El umbral es más agresivo que el estándar (0.50) porque bloquear erróneamente un legítimo es 4× menos dañino que aprobar un fraude.
> Esto a nivel teórico puede plantearse, sin embargo puede utilizarce un TH óptimo según lo mencionado anteriormente combinado con diferentes rangos de montos, u otra variable que pueda ser de utilidad

****

## Desarrollo de la solución

A continuación, se describen los pasos a implementar para dar solución al problema planteado, teniendo en cuenta la función de ganancia a optimizar

### Análisis exploratorio de datos EDA

En este apartado, se realiza un análisis exploratorio de datos, dónde se determine de manera concisa, la estructura general del dataset, las variables a utilizar, estadisticos basicos y entre otros que ayuden a dar contexto del dataset a utilizar, se proponen los siguientes puntos base:

- Definición de Train/Test y particionamiento de datos
- Información general del dataset `MercadoLibre Data Scientist Hiring Test - Fraud Dataset with date v2.csv`.
- Estadisticos basicos como promedios, varianzas, correlaciones, relaciones entre variables y anomalías que se puedan identificar, variables categóricas y entre otras.
- Outliers, nulidad y ceros.
- Depuración de variables que funcionen como id's.
- Distribución del target.

### Feature engineering

En este apartado, se realiza el feature engineering pertinente sobre los hallazgos del EDA, es decir, que transformaciones serán necesarias y que técnicas son pertinentes de utilizar, tanto para las variables como el desbalanceo que pueda existir:

- Posibles técnicas apalicar para el preprocesamiento de la información.
- Elección de variables candidatas y depuración de las mismas.
- Imputación de valores faltantes o missing values.
- Construcción de nuevas features (ratios o agregaciones).
- Determinar desbalanceo y aplicar técnicas que lo compensen.
- Pruebas estadísticas para datos categóricos y numéricos.
- Selección final de features.

### Modelación, evaluación y optimización

Por último en este apartado, se realzia la modelación, evaluación y optimización del modelo final, además de las experimentaciones pertinentes, resultados finales y next steps que s epeudan considerar:

- Algoritmos a implementar (XGBoost, CatBoost, Random Forest, LightGBM y entre otros a elección).
- Evaluación de modelos.
- Comparación de resultados y elección de modelo final.
- Optimización de Threshold e implementación de la función de ganancia.
- Conclusiones finales y próximos pasos.